# HU-01 · Registro

[Inicio](../../tests.ipynb) · Anterior: [Índice de QA de Typr](../../tests.ipynb) · Siguiente: [HU-02 · Práctica individual](hu02_practica.ipynb)

Las fichas incluyen los criterios del Plan Maestro. Las referencias U/G/F/W/D y las rutas abreviadas en **Evidencia** se interpretan en la [matriz maestra](../../docs/test/qa-master-matrix.md). Una ejecución local no implica un resultado publicado ni prueba un control de entrega.

## TC-FUNC-01

- **Resumen de la ficha:** Crear la cuenta, iniciar sesión y llegar al panel; validar usuario de 2, 3, 20 y 21 caracteres y contraseña de 7 y 8. No devolver contraseña ni hash.
- **Método previsto:** Manual + API.
- **Estado:** **V — Verificado**.
- **Evidencia:** En la matriz (**V**): Chromium local con la API actual: 6 escenarios y 33 pasos aprobados en `@registration_boundaries` ([registro en navegador](../../frontend/e2e/registration.feature)). [test_registration_validation.py](../../backend/tests/test_registration_validation.py) verifica el cuerpo del HTTP 422 sin datos enviados; [registro en backend](../../backend/tests/features/registration.feature) (`@RI-02`) verifica la omisión de contraseña y hash en registro, inicio de sesión y perfil.
- **Brecha o límite:** No se identificó una brecha para el escenario y entorno documentados.

**Criterio de aceptación literal** · Plan §5.5 → [HU-01 #8](https://github.com/1Bnja/Typr/issues/8).

```gherkin
Característica: Registro e inicio de sesión

  Escenario: Registro exitoso
    Dado que el visitante se encuentra en la pantalla de registro
    Y completa los campos de usuario, correo y contraseña con datos válidos
    Cuando presiona el botón "Crear cuenta"
    Entonces el sistema crea la cuenta del usuario
    Y autentica automáticamente la sesión
    Y redirige al usuario al dashboard
```

**Procedimiento del Plan (§5.1)**

Protocolo complementario para TC-FUNC-01 y TC-FUNC-02; no reemplaza los escenarios de HU-01.

- **Preparación:** Pantalla de registro abierta; para el segundo escenario ya existe `usuario@ejemplo.com`.
- **Pasos:** Completar datos válidos y presionar «Crear cuenta»; repetir con el correo duplicado; repetir el flujo válido con usuarios de 2, 3, 20 y 21 caracteres y contraseñas de 7 y 8.
- **Resultado:** Alta, autenticación y redirección al dashboard; para el duplicado no hay alta y aparece «Este correo ya está en uso»; los valores fuera de rango se rechazan junto al campo.
- **Éxito:** Se cumplen todos los Entonces y la respuesta nunca incluye contraseña ni hash.

<details>
<summary>Texto literal de la ficha del Plan</summary>

HU-01 Registro e inicio de sesión · POST /api/v1/auth/register + vista Login/Registro Funcional · escenarios Gherkin (happy path y error) con valores límite Alta (P2) / Mayor Manual (navegador) y automatizada (API con unittest + TestClient) Escenario 1: el visitante se encuentra en la pantalla de registro. Escenario 2: existe una cuenta registrada con el correo usuario@ejemplo.com. 1. Completa usuario, correo y contraseña válidos y presiona “Crear cuenta”. 2. Completa el formulario con usuario@ejemplo.com y presiona “Crear cuenta”. 3. Repite el paso 1 con valores límite: usuario de 2, 3, 20 y 21 caracteres; contraseña de 7 y 8. Usuario qa user · correos qa user@typr.test y usuario@ejemplo.com · _ _ contraseña válida de 8 caracteres 1. Crea la cuenta, autentica la sesión y redirige al dashboard. 2. No crea la cuenta y muestra “Este correo ya está en uso”. 3. Los valores fuera de rango se rechazan con un mensaje junto al campo. Se cumplen todos los Entonces de ambos escenarios y la respuesta nunca incluye la contraseña ni su hash.

</details>

**Gherkin ejecutable (original en inglés)** · [Fuente](../../backend/tests/features/registration.feature).

```gherkin
@auth @postgresql
Feature: Registration preserves unique account identities and protects passwords

  @TC-FUNC-01 @RI-02
  Scenario: Passwords are stored as bcrypt and never exposed by account endpoints
    When a new account registers, logs in, and requests its profile
    Then its persisted password is a bcrypt hash of the submitted password
    And the register, login, and profile responses omit password material
```

**Gherkin ejecutable (original en inglés)** · [Fuente](../../frontend/e2e/session.feature).

```gherkin
@TC-AUTO-01 @RI-14
Feature: Browser session lifecycle
  Background:
    Given I register a new account in the browser

  @TC-FUNC-01
  Scenario: Reload, sign out, go Back and sign in again
    When I reload the profile and the API validates my session
    When I sign out with another tab open and a separate session for my account
    Then only the browser session is revoked
    And the other tab detects the revocation on a protected request and on reload
    Then I can go Back without seeing private data and sign in with a new session
```

**Gherkin ejecutable (original en inglés)** · [Fuente](../../frontend/e2e/registration.feature).

```gherkin
Feature: Registration validation

  @TC-FUNC-01 @registration_boundaries
  Scenario Outline: Registration accepts a <field> of <length> characters
    Given I open the registration form
    When I enter a <field> of <length> characters and submit valid remaining fields
    Then registration creates an account and opens the profile

    Examples:
      | field    | length |
      | username | 3      |
      | username | 20     |
      | password | 8      |

  @TC-FUNC-01 @registration_boundaries
  Scenario Outline: Registration rejects a <field> of <length> characters at the field
    Given I open the registration form
    When I enter a <field> of <length> characters and submit valid remaining fields
    Then the <field> field displays a length message mentioning <limit> characters
    And registration keeps all my entered values

    Examples:
      | field    | length | limit |
      | username | 2      | 3     |
      | username | 21     | 20    |
      | password | 7      | 8     |
```

## TC-FUNC-02

- **Resumen de la ficha:** Rechazar el correo duplicado con 409, sin crear otra cuenta y con aviso junto al campo.
- **Método previsto:** Manual + API.
- **Estado:** **V — Verificado**.
- **Evidencia:** En la matriz (**V**): [registro en navegador](../../frontend/e2e/registration.feature) (`@TC-FUNC-02`) comprueba 409, mensaje exacto junto al correo y conservación de valores, incluido el conflicto doble. [Registro en backend](../../backend/tests/features/registration.feature) (`@RI-01`, ejemplo `email` de `Overlapping registrations collide on <identity>`) comprueba 201/409 y una sola cuenta persistida en PostgreSQL para solicitudes simultáneas.
- **Brecha o límite:** No se identificó una brecha para el escenario y entorno documentados.

**Criterio de aceptación literal** · Plan §5.5 → [HU-01 #8](https://github.com/1Bnja/Typr/issues/8).

```gherkin
Característica: Registro e inicio de sesión

  Escenario: Intento de registro con correo duplicado
    Dado que existe una cuenta registrada con el correo "usuario@ejemplo.com"
    Y el visitante completa el formulario con el correo "usuario@ejemplo.com"
    Cuando presiona el botón "Crear cuenta"
    Entonces el sistema no crea la cuenta
    Y muestra el mensaje de error "Este correo ya está en uso"
```

El [procedimiento conjunto de §5.1](#TC-FUNC-01) incluye la comprobación adicional de la respuesta.

**Gherkin ejecutable (original en inglés)** · [Fuente](../../frontend/e2e/registration.feature).

```gherkin
Feature: Registration validation

  @TC-FUNC-02
  Scenario Outline: Email conflicts are reported at the email field
    Given an account exists and I open the registration form
    When I submit registration with a <conflict> conflict
    Then the email field reports "Este correo ya está en uso"
    And registration keeps all my entered values

    Examples:
      | conflict |
      | email    |
      | dual     |

  @TC-FUNC-02
  Scenario: A username-only conflict keeps the existing form error
    Given an account exists and I open the registration form
    When I submit registration with a username conflict
    Then the form reports "El usuario ya existe bro" without marking email invalid
    And registration keeps all my entered values
```

El ejecutable añade el HTTP 409 y la presentación de conflictos de usuario; estos no forman parte del texto literal de HU-01.

## Reproducción local opcional

Prepare el entorno desechable de la [guía de pruebas](../../test/README.md). Ejecute solo la celda que necesita, con el kernel iniciado en la raíz de Typr o dentro de `notebooks/`. No ejecute en paralelo. Abrir el cuaderno no inicia pruebas; la salida describe una corrida local, no cambia el estado de las fichas. Chromium se abre en una sesión gráfica local.

### Criterios de aceptación · TC-FUNC-01 y TC-FUNC-02

**Plan (§5.1):** El registro válido crea y autentica la cuenta y abre el panel; los valores fuera de rango muestran un error junto al campo. El correo duplicado no crea otra cuenta y muestra «Este correo ya está en uso». La API no devuelve contraseña ni hash.

**Qué comprueba esta celda:** Chromium filtra ambos casos. Comprueba HTTP 201 y llegada al panel para valores válidos (usuario de 3 y 20 caracteres; contraseña de 8), HTTP 422 con aviso de campo para los inválidos (usuario de 2 y 21; contraseña de 7), y HTTP 409 con conservación de los datos del formulario en los conflictos. Distingue el aviso junto al correo del error general por nombre de usuario; la etiqueta también incluye el flujo completo de sesión.

**Límite:** El HTTP 409 y la interfaz no sustituyen el recuento de cuentas en PostgreSQL ni la comprobación de que la API omite contraseña y hash.

In [ ]:
from pathlib import Path
from runpy import run_path

directorio = Path.cwd().resolve()
for raiz in (directorio, *directorio.parents):
    if (raiz / "docker-compose.test.yml").is_file() and (raiz / "notebooks/ejecutar_qa.py").is_file():
        break
else:
    raise RuntimeError("Inicie el kernel desde la raíz de Typr o dentro de notebooks/.")

run_path(str(raiz / "notebooks/ejecutar_qa.py"))["ejecutar"]('browser-gherkin', etiquetas='@TC-FUNC-01 or @TC-FUNC-02')

[Inicio](../../tests.ipynb) · Anterior: [Índice de QA de Typr](../../tests.ipynb) · Siguiente: [HU-02 · Práctica individual](hu02_practica.ipynb)